# OOF Error Analysis, LIME/DiCE Configuration, and Weight Identification

This notebook records the full path from out-of-fold (OOF) predictions
to false-negative explanation aggregation, DiCE confirmation, and the
exact sample-weight rules used by the model-refinement evaluation.

The deterministic OOF and weight-generation cells run directly.
Full LIME and DiCE generation is disabled by default because it is
computationally expensive; set the two run flags to `True` to regenerate
those explanations. The released summaries are loaded otherwise.


## 1. Connect Google Drive and install the environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

configured_root = os.environ.get("LICE_PROJECT_ROOT")
project_candidates = [Path(configured_root) if configured_root else None, Path.cwd(), Path("/content/drive/MyDrive/Research/LICE_Guided_Model_Refinement_v1.3.0"), Path("/content/drive/MyDrive/LICE_Guided_Model_Refinement_v1.3.0")]
PROJECT_ROOT = next((p.resolve() for p in project_candidates if p and (p / "data").exists()), None)
assert PROJECT_ROOT is not None, "Repository not found; set LICE_PROJECT_ROOT."

from importlib.metadata import PackageNotFoundError, version
import sys

PINNED_BINARY_STACK = {
    "numpy": ("numpy", "2.2.6"),
    "pandas": ("pandas", "2.3.3"),
    "scikit-learn": ("sklearn", "1.5.2"),
    "scipy": ("scipy", "1.15.3"),
    "statsmodels": ("statsmodels", "0.14.4"),
}

def installed_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None

restart_required = any(
    installed_version(distribution) != required
    or (
        module in sys.modules
        and getattr(sys.modules[module], "__version__", None) != required
    )
    for distribution, (module, required) in PINNED_BINARY_STACK.items()
)

%pip install -q -r "{PROJECT_ROOT / 'environment' / 'requirements_oof.txt'}"

if restart_required:
    print(
        "Pinned packages were installed. Colab will restart now. "
        "After it reconnects, run this setup cell once more and "
        "then continue to the next cell.",
        flush=True,
    )
    os.kill(os.getpid(), 9)

print(f"Release folder: {PROJECT_ROOT}")


## 2. Imports and experiment constants


In [ ]:
from pathlib import Path
import gzip
import hashlib
import json
import random

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

try:
    from IPython.display import display
except ImportError:
    display = print

DATA_PATH = PROJECT_ROOT / "data" / "diabetes_brfss2015_prepared.csv"
OOF_RESULTS_DIR = PROJECT_ROOT / "results" / "oof"
ABLATION_RESULTS_DIR = PROJECT_ROOT / "results" / "test_ablation"
OOF_PREDICTIONS_DIR = PROJECT_ROOT / "predictions" / "oof"
INPUT_DIR = PROJECT_ROOT / "input_artifacts"
SPLITS_DIR = PROJECT_ROOT / "splits"
for directory in (OOF_RESULTS_DIR, ABLATION_RESULTS_DIR, OOF_PREDICTIONS_DIR, INPUT_DIR, SPLITS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20
OOF_FOLDS = 3
DECISION_THRESHOLD = 0.50
MODEL_PARAMETERS = {
    "ccp_alpha": 0.0,
    "criterion": "friedman_mse",
    "init": None,
    "learning_rate": 0.1,
    "loss": "log_loss",
    "max_depth": 5,
    "max_features": None,
    "max_leaf_nodes": None,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "n_estimators": 100,
    "n_iter_no_change": None,
    "random_state": RANDOM_SEED,
    "subsample": 1.0,
    "tol": 0.0001,
    "validation_fraction": 0.1,
    "verbose": 0,
    "warm_start": False,
}

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()

def write_deterministic_gzip_csv(frame, path):
    with Path(path).open("wb") as raw_stream:
        with gzip.GzipFile(
            filename="", mode="wb", fileobj=raw_stream,
            compresslevel=9, mtime=0,
        ) as gzip_stream:
            frame.to_csv(gzip_stream, index=False, float_format="%.17g")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 17)


## 3. Load the fixed training split


In [ ]:
data = pd.read_csv(DATA_PATH)
X = data.drop(columns="Outcome")
y = data["Outcome"].astype(int)
source_indices = np.arange(len(data), dtype=int)
train_idx, test_idx = train_test_split(
    source_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=None,
)
X_train = X.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)

assert len(X_train) == 55245
assert len(test_idx) == 13812
print("Training rows:", len(X_train))


## 4. OOF predictions and false-negative positions

`StratifiedKFold(n_splits=3, shuffle=False)` assigns each training row
to one validation fold. Each fold model is fitted only on the other two
folds. Predictions use the classifier's class prediction, corresponding
to the fixed probability threshold of 0.50 for this binary classifier.


In [ ]:
cv = StratifiedKFold(n_splits=OOF_FOLDS, shuffle=False)
oof_prediction = np.full(len(X_train), -1, dtype=int)
oof_probability = np.full(len(X_train), np.nan, dtype=float)
validation_fold = np.full(len(X_train), -1, dtype=int)
fold_rows = []

for fold, (fit_positions, validation_positions) in enumerate(
    cv.split(X_train, y_train), start=1
):
    model = GradientBoostingClassifier(**MODEL_PARAMETERS)
    model.fit(
        X_train.iloc[fit_positions],
        y_train.iloc[fit_positions],
    )
    probability = model.predict_proba(
        X_train.iloc[validation_positions]
    )[:, 1]
    prediction = model.predict(X_train.iloc[validation_positions])
    oof_probability[validation_positions] = probability
    oof_prediction[validation_positions] = prediction
    validation_fold[validation_positions] = fold

    truth = y_train.iloc[validation_positions].to_numpy()
    tn, fp, fn, tp = confusion_matrix(truth, prediction).ravel()
    fold_rows.append({
        "Fold": fold,
        "Validation_Rows": len(validation_positions),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Accuracy": accuracy_score(truth, prediction),
        "Precision": precision_score(truth, prediction),
        "Recall": recall_score(truth, prediction),
        "F1": f1_score(truth, prediction),
    })

assert np.all(oof_prediction >= 0)
oof_table = pd.DataFrame({
    "Source_Row_Index": train_idx,
    "Train_Position": np.arange(len(X_train)),
    "OOF_Validation_Fold": validation_fold,
    "y_true": y_train.to_numpy(),
    "oof_prediction": oof_prediction,
    "oof_probability": oof_probability,
})
oof_table["Error_Type"] = np.select(
    [
        (oof_table.y_true == 1) & (oof_table.oof_prediction == 0),
        (oof_table.y_true == 0) & (oof_table.oof_prediction == 1),
    ],
    ["FN", "FP"],
    default="Correct",
)

write_deterministic_gzip_csv(
    oof_table,
    OOF_PREDICTIONS_DIR / "baseline_oof_predictions_full_precision.csv.gz",
)
write_deterministic_gzip_csv(
    oof_table.loc[oof_table.Error_Type != "Correct"],
    OOF_PREDICTIONS_DIR / "oof_error_indices_full_precision.csv.gz",
)
fold_metrics = pd.DataFrame(fold_rows)
fold_metrics.to_csv(
    OOF_RESULTS_DIR / "oof_fold_metrics_full_precision.csv",
    index=False,
    float_format="%.17g",
)

tn, fp, fn, tp = confusion_matrix(y_train, oof_prediction).ravel()
assert (int(fp), int(fn)) == (8381, 5698)
oof_summary = pd.DataFrame([{
    "Rows": len(oof_table), "TN": int(tn), "FP": int(fp),
    "FN": int(fn), "TP": int(tp),
    "Accuracy": accuracy_score(y_train, oof_prediction),
    "Precision": precision_score(y_train, oof_prediction),
    "Recall": recall_score(y_train, oof_prediction),
    "F1": f1_score(y_train, oof_prediction),
}])
oof_summary.to_csv(
    OOF_RESULTS_DIR / "oof_summary_full_precision.csv",
    index=False,
    float_format="%.17g",
)
display(oof_summary)


## 5. Feature definitions used by LIME and DiCE

`Immutable` features are never proposed as actions. `Health-usable`
features may be used for explanation aggregation and DiCE. `Caution`
features describe established diagnoses. `Policy-excluded` and
`Other-excluded` features are not used to define interventions.


In [ ]:
FEATURE_DEFINITIONS = {
    "HighBP": "Health-usable", "HighChol": "Health-usable",
    "CholCheck": "Health-usable", "BMI": "Health-usable",
    "Smoker": "Health-usable", "Stroke": "Caution",
    "HeartDiseaseorAttack": "Caution", "PhysActivity": "Health-usable",
    "Fruits": "Health-usable", "Veggies": "Health-usable",
    "HvyAlcoholConsump": "Health-usable", "AnyHealthcare": "Other-excluded",
    "NoDocbcCost": "Other-excluded", "GenHlth": "Health-usable",
    "MentHlth": "Health-usable", "PhysHlth": "Health-usable",
    "DiffWalk": "Health-usable", "Sex": "Immutable", "Age": "Immutable",
    "Education": "Policy-excluded", "Income": "Policy-excluded",
}
ACTIONABLE_DICE_FEATURES = ["GenHlth", "HighBP", "HighChol", "BMI"]
CONTINUOUS_DICE_FEATURES = ["BMI", "MentHlth", "PhysHlth"]
display(
    pd.Series(FEATURE_DEFINITIONS, name="Definition")
    .rename_axis("Feature").reset_index()
)


## 6. LIME configuration and false-negative aggregation

Configuration: tabular classification mode; training-data
background; quartile discretization; Euclidean distance; kernel
width $0.75\sqrt{21}$; five explanation terms; 5,000 perturbed
samples; and per-case seed `42 + Train_Position`.

A historical working notebook did not record a LIME random seed.
The public implementation fixes the seed above for deterministic
reruns while retaining the released four patterns used downstream.
Aggregation is over OOF false negatives only. Each cleaned
`(Feature, Condition)` term is scored by
`case coverage × mean absolute LIME coefficient`; eligible negative
terms are ordered by score, frequency, mean absolute coefficient,
feature, and condition, and the first four are selected.

When full regeneration is enabled, the regenerated four patterns
are compared explicitly with the released pattern artifact. A
mismatch stops the run instead of silently replacing the reported
selection.


In [ ]:
RUN_FULL_LIME = False
LIME_CONFIGURATION = {
    "mode": "classification",
    "discretize_continuous": True,
    "discretizer": "quartile",
    "distance_metric": "euclidean",
    "kernel_width": float(0.75 * np.sqrt(X_train.shape[1])),
    "num_features": 5,
    "num_samples": 5000,
    "seed_rule": "42 + Train_Position",
    "explained_cases": "OOF false negatives only",
}

reported_patterns = pd.read_csv(
    ABLATION_RESULTS_DIR / "selected_lime_patterns_reported.csv"
)

def normalize_lime_condition(value):
    return " ".join(str(value).replace(".00", "").split())

if RUN_FULL_LIME:
    from lime.lime_tabular import LimeTabularExplainer
    from tqdm.auto import tqdm

    explanation_rows = []
    false_negative_positions = oof_table.loc[
        oof_table.Error_Type == "FN", "Train_Position"
    ].to_numpy(dtype=int)

    for position in tqdm(false_negative_positions):
        explainer = LimeTabularExplainer(
            training_data=X_train.to_numpy(),
            feature_names=X_train.columns.tolist(),
            class_names=["No diabetes", "Diabetes"],
            mode="classification",
            discretize_continuous=True,
            discretizer="quartile",
            kernel_width=LIME_CONFIGURATION["kernel_width"],
            random_state=RANDOM_SEED + int(position),
        )
        fold = int(validation_fold[position])
        fit_positions = np.flatnonzero(validation_fold != fold)
        fold_model = GradientBoostingClassifier(**MODEL_PARAMETERS)
        fold_model.fit(
            X_train.iloc[fit_positions],
            y_train.iloc[fit_positions],
        )
        explanation = explainer.explain_instance(
            X_train.iloc[position].to_numpy(),
            fold_model.predict_proba,
            num_features=LIME_CONFIGURATION["num_features"],
            num_samples=LIME_CONFIGURATION["num_samples"],
            distance_metric=LIME_CONFIGURATION["distance_metric"],
        )
        for term, coefficient in explanation.as_list(label=1):
            feature = next(
                (name for name in X_train.columns if name in term),
                None,
            )
            explanation_rows.append({
                "Source_Row_Index": int(train_idx[position]),
                "Train_Position": int(position),
                "Feature": feature,
                "Condition": term,
                "LIME_Coefficient": float(coefficient),
            })

    lime_long = pd.DataFrame(explanation_rows)
    write_deterministic_gzip_csv(
        lime_long,
        ABLATION_RESULTS_DIR / "lime_fn_explanations_full_precision.csv.gz",
    )
    lime_long["Absolute_Coefficient"] = (
        lime_long.LIME_Coefficient.abs()
    )
    aggregated = (
        lime_long.groupby(["Feature", "Condition"], as_index=False)
        .agg(
            Case_Frequency=("Train_Position", "nunique"),
            Mean_LIME_Coefficient=("LIME_Coefficient", "mean"),
            Mean_Absolute_LIME=("Absolute_Coefficient", "mean"),
        )
    )
    aggregated["Case_Coverage"] = (
        aggregated.Case_Frequency / len(false_negative_positions)
    )
    aggregated["Priority_Score"] = (
        aggregated.Case_Coverage
        * aggregated.Mean_Absolute_LIME
    )
    aggregated["Feature_Definition"] = (
        aggregated.Feature.map(FEATURE_DEFINITIONS)
    )
    eligible = aggregated.loc[
        (aggregated.Mean_LIME_Coefficient < 0)
        & (aggregated.Feature_Definition == "Health-usable")
    ].sort_values(
        [
            "Priority_Score", "Case_Frequency",
            "Mean_Absolute_LIME", "Feature", "Condition",
        ],
        ascending=[False, False, False, True, True],
    )
    selected_patterns = eligible.head(4).reset_index(drop=True)
    selected_patterns.to_csv(
        ABLATION_RESULTS_DIR / "selected_lime_patterns_regenerated.csv",
        index=False,
        float_format="%.17g",
    )

    lime_comparison = pd.DataFrame({
        "Rank": np.arange(1, 5),
        "Reported_Feature": reported_patterns.Feature.to_numpy(),
        "Reported_Condition": (
            reported_patterns.LIME_Condition.to_numpy()
        ),
        "Regenerated_Feature": (
            selected_patterns.Feature.to_numpy()
        ),
        "Regenerated_Condition": (
            selected_patterns.Condition.to_numpy()
        ),
    })
    lime_comparison["Feature_Match"] = (
        lime_comparison.Reported_Feature
        == lime_comparison.Regenerated_Feature
    )
    lime_comparison["Condition_Match"] = [
        normalize_lime_condition(reported)
        == normalize_lime_condition(regenerated)
        for reported, regenerated in zip(
            lime_comparison.Reported_Condition,
            lime_comparison.Regenerated_Condition,
        )
    ]
    lime_comparison["Comparison_Status"] = "Regenerated"
    lime_comparison.to_csv(
        ABLATION_RESULTS_DIR / "lime_reported_vs_regenerated.csv",
        index=False,
    )
    assert lime_comparison[
        ["Feature_Match", "Condition_Match"]
    ].to_numpy().all(), (
        "Regenerated LIME selection differs from the released "
        "pattern artifact; inspect lime_reported_vs_regenerated.csv."
    )
else:
    selected_patterns = reported_patterns.copy()
    lime_comparison = reported_patterns[
        ["Pattern_ID", "Feature", "LIME_Condition"]
    ].copy()
    lime_comparison["Comparison_Status"] = (
        "Reported artifact loaded; full regeneration disabled"
    )

display(LIME_CONFIGURATION)
display(selected_patterns)
display(lime_comparison)


## 7. DiCE confirmation configuration

DiCE uses the random method, the fitted fold-specific GBDT,
`desired_class=1`, three counterfactuals per OOF false negative,
and the per-case seed `42 + Train_Position`. The permitted
features are `GenHlth`, `HighBP`, `HighChol`, and `BMI`; all other
features are held fixed.

Feature confirmation is a **case-level** criterion: a feature is
confirmed when at least 30% of all 5,698 OOF false-negative cases
have at least one generated counterfactual in which that feature
changes. It is not based on the fraction of individual
counterfactual rows. A case-level combination is retained when at
least two features change in at least 50% of successful cases.

When full regeneration is enabled, regenerated feature coverage,
confirmed-feature membership, successful-case counts, and the
two-or-more-change count are compared explicitly with the released
DiCE artifacts.


In [ ]:
RUN_FULL_DICE = False
DICE_CONFIGURATION = {
    "method": "random",
    "total_CFs": 3,
    "desired_class": 1,
    "features_to_vary": ACTIONABLE_DICE_FEATURES,
    "continuous_features": CONTINUOUS_DICE_FEATURES,
    "seed_rule": "42 + Train_Position",
    "feature_confirmation_threshold": 0.30,
    "feature_confirmation_unit": (
        "percentage of all OOF false-negative cases with at "
        "least one change in the feature"
    ),
    "case_combination_threshold": 0.50,
    "minimum_changed_features_per_case": 2,
}

reported_dice_summary = json.loads(
    (ABLATION_RESULTS_DIR / "dice_confirmation_summary.json").read_text()
)
reported_feature_confirmation = pd.read_csv(
    ABLATION_RESULTS_DIR / "dice_feature_confirmation_reported.csv"
)

if RUN_FULL_DICE:
    import dice_ml
    from tqdm.auto import tqdm

    false_negative_positions = oof_table.loc[
        oof_table.Error_Type == "FN", "Train_Position"
    ].to_numpy(dtype=int)
    change_rows = []
    status_rows = []

    for position in tqdm(false_negative_positions):
        fold = int(validation_fold[position])
        fit_positions = np.flatnonzero(validation_fold != fold)
        fold_model = GradientBoostingClassifier(**MODEL_PARAMETERS)
        fold_model.fit(
            X_train.iloc[fit_positions],
            y_train.iloc[fit_positions],
        )
        dice_frame = X_train.iloc[fit_positions].copy()
        dice_frame["Outcome"] = (
            y_train.iloc[fit_positions].to_numpy()
        )
        data_interface = dice_ml.Data(
            dataframe=dice_frame,
            continuous_features=CONTINUOUS_DICE_FEATURES,
            outcome_name="Outcome",
        )
        model_interface = dice_ml.Model(
            model=fold_model, backend="sklearn"
        )
        dice = dice_ml.Dice(
            data_interface, model_interface, method="random"
        )
        np.random.seed(RANDOM_SEED + int(position))
        random.seed(RANDOM_SEED + int(position))
        query = X_train.iloc[[position]].copy()
        try:
            generated = dice.generate_counterfactuals(
                query,
                total_CFs=DICE_CONFIGURATION["total_CFs"],
                desired_class=DICE_CONFIGURATION["desired_class"],
                features_to_vary=(
                    DICE_CONFIGURATION["features_to_vary"]
                ),
            )
            cf_frame = (
                generated.cf_examples_list[0].final_cfs_df
            )
        except Exception as error:
            status_rows.append({
                "Train_Position": int(position),
                "Status": "No counterfactual",
                "Detail": str(error),
            })
            continue
        if cf_frame is None or cf_frame.empty:
            status_rows.append({
                "Train_Position": int(position),
                "Status": "No counterfactual",
                "Detail": "",
            })
            continue
        status_rows.append({
            "Train_Position": int(position),
            "Status": "Successful",
            "Detail": "",
        })
        original = query.iloc[0]
        for cf_number, (_, counterfactual) in enumerate(
            cf_frame.iterrows(), start=1
        ):
            for feature in ACTIONABLE_DICE_FEATURES:
                original_value = float(original[feature])
                counterfactual_value = float(
                    counterfactual[feature]
                )
                changed = not np.isclose(
                    counterfactual_value, original_value
                )
                change_rows.append({
                    "Source_Row_Index": int(train_idx[position]),
                    "Train_Position": int(position),
                    "Counterfactual": cf_number,
                    "Feature": feature,
                    "Changed": bool(changed),
                    "Original_Value": original_value,
                    "Counterfactual_Value": counterfactual_value,
                    "Delta": (
                        counterfactual_value - original_value
                    ),
                })

    changes = pd.DataFrame(change_rows)
    statuses = pd.DataFrame(status_rows)
    write_deterministic_gzip_csv(
        changes,
        ABLATION_RESULTS_DIR / "dice_changes_full_precision.csv.gz",
    )
    statuses.to_csv(
        ABLATION_RESULTS_DIR / "dice_case_status.csv", index=False
    )

    total_fn_cases = len(false_negative_positions)
    successful_cases = statuses.loc[
        statuses.Status == "Successful", "Train_Position"
    ].nunique()
    case_feature_changes = (
        changes.groupby(
            ["Train_Position", "Feature"], as_index=False
        ).Changed.max()
    )
    case_counts = (
        case_feature_changes.groupby("Feature", as_index=False)
        .Changed.sum()
        .rename(columns={"Changed": "Cases_With_Change"})
    )
    cf_counts = (
        changes.groupby("Feature", as_index=False)
        .agg(
            Changed_CF_Rows=("Changed", "sum"),
            Total_CF_Feature_Rows=("Changed", "size"),
        )
    )
    feature_confirmation = cf_counts.merge(
        case_counts, on="Feature", how="left"
    )
    feature_confirmation["Successful_FN_Cases"] = (
        successful_cases
    )
    feature_confirmation["Total_FN_Cases"] = total_fn_cases
    feature_confirmation[
        "Case_Level_Coverage_Percentage_AllFN"
    ] = (
        feature_confirmation.Cases_With_Change
        / total_fn_cases
        * 100.0
    )
    feature_confirmation[
        "Case_Level_Coverage_Percentage_SuccessOnly"
    ] = (
        feature_confirmation.Cases_With_Change
        / successful_cases
        * 100.0
    )
    feature_confirmation["Confirmed"] = (
        feature_confirmation[
            "Case_Level_Coverage_Percentage_AllFN"
        ]
        >= 100.0 * DICE_CONFIGURATION["feature_confirmation_threshold"]
    )
    feature_confirmation.to_csv(
        ABLATION_RESULTS_DIR
        / "dice_feature_confirmation_regenerated.csv",
        index=False,
        float_format="%.17g",
    )

    qualifying_cases = int((
        case_feature_changes.groupby("Train_Position").Changed.sum()
        >= DICE_CONFIGURATION[
            "minimum_changed_features_per_case"
        ]
    ).sum())
    case_fraction = qualifying_cases / successful_cases
    confirmed_features = sorted(
        feature_confirmation.loc[
            feature_confirmation.Confirmed, "Feature"
        ].tolist()
    )
    same_feature_set = (
        sorted(confirmed_features)
        == sorted(selected_patterns["Feature"].tolist())
    )
    combination_rule_satisfied = (
        case_fraction
        >= DICE_CONFIGURATION["case_combination_threshold"]
    )
    retain_existing_weight_rule = (
        same_feature_set and combination_rule_satisfied
    )

    dice_summary = {
        "same_feature_set": bool(same_feature_set),
        "combination_rule_satisfied": bool(combination_rule_satisfied),
        "retain_existing_weight_rule": bool(retain_existing_weight_rule),
        "total_oof_false_negative_cases": int(total_fn_cases),
        "successful_false_negative_cases": int(successful_cases),
        "no_counterfactual_cases": int(
            total_fn_cases - successful_cases
        ),
        "cases_with_two_or_more_feature_changes": int(
            qualifying_cases
        ),
        "percentage_successful_cases_with_two_or_more_changes": (
            float(case_fraction * 100.0)
        ),
        "dice_confirmed_features": confirmed_features,
        "configuration": DICE_CONFIGURATION,
    }
    with (
        ABLATION_RESULTS_DIR
        / "dice_confirmation_summary_regenerated.json"
    ).open("w") as stream:
        json.dump(dice_summary, stream, indent=2)

    dice_comparison = reported_feature_confirmation[
        [
            "Feature",
            "Case_Level_Change_Coverage_Percentage_AllFN",
        ]
    ].rename(columns={
        "Case_Level_Change_Coverage_Percentage_AllFN": (
            "Reported_Case_Level_Coverage_Percentage_AllFN"
        )
    }).merge(
        feature_confirmation[
            [
                "Feature",
                "Case_Level_Coverage_Percentage_AllFN",
                "Confirmed",
            ]
        ].rename(columns={
            "Case_Level_Coverage_Percentage_AllFN": (
                "Regenerated_Case_Level_Coverage_Percentage_AllFN"
            ),
            "Confirmed": "Regenerated_Confirmed",
        }),
        on="Feature",
        how="outer",
    )
    dice_comparison["Reported_Confirmed"] = (
        dice_comparison[
            "Reported_Case_Level_Coverage_Percentage_AllFN"
        ]
        >= 100.0 * DICE_CONFIGURATION[
        "feature_confirmation_threshold"]
    )
    dice_comparison["Coverage_Difference_Percentage_Points"] = (
        dice_comparison[
            "Regenerated_Case_Level_Coverage_Percentage_AllFN"
        ]
        - dice_comparison[
            "Reported_Case_Level_Coverage_Percentage_AllFN"
        ]
    )
    dice_comparison["Confirmation_Match"] = (
        dice_comparison.Reported_Confirmed
        == dice_comparison.Regenerated_Confirmed
    )
    dice_comparison["Comparison_Status"] = "Regenerated"
    dice_comparison.to_csv(
        ABLATION_RESULTS_DIR / "dice_reported_vs_regenerated.csv",
        index=False,
        float_format="%.17g",
    )

    assert dice_comparison.Confirmation_Match.all(), (
        "Regenerated DiCE confirmed-feature set differs from "
        "the released artifact."
    )
    assert successful_cases == int(
        reported_dice_summary[
            "successful_false_negative_cases"
        ]
    )
    assert qualifying_cases == int(
        reported_dice_summary[
            "cases_with_two_or_more_feature_changes"
        ]
    )
else:
    dice_summary = reported_dice_summary
    feature_confirmation = reported_feature_confirmation.copy()
    dice_comparison = feature_confirmation[
        [
            "Feature",
            "Case_Level_Change_Coverage_Percentage_AllFN",
        ]
    ].copy()
    dice_comparison["Reported_Confirmed"] = (
        dice_comparison[
            "Case_Level_Change_Coverage_Percentage_AllFN"
        ]
        >= 100.0 * DICE_CONFIGURATION[
        "feature_confirmation_threshold"]
    )
    dice_comparison["Comparison_Status"] = (
        "Reported artifact loaded; full regeneration disabled"
    )

display(DICE_CONFIGURATION)
display(dice_summary)
display(feature_confirmation)
display(dice_comparison)


## 8. Exact pattern matching and sample-weight assignment

The four released conditions are evaluated on every training row:
`GenHlth <= 2`, `HighBP <= 0`, `HighChol <= 0`, and `BMI <= 25`.
Only positive-class training rows receive increased weights. Rows
matching exactly two patterns receive Mild/Balanced/High weights of
1.10/1.15/1.20. Rows matching three or four receive
1.15/1.25/1.35. All other rows retain weight 1.0.


In [ ]:
PATTERN_RULES = {
    "GenHlth <= 2": lambda frame: frame["GenHlth"] <= 2,
    "HighBP <= 0": lambda frame: frame["HighBP"] <= 0,
    "HighChol <= 0": lambda frame: frame["HighChol"] <= 0,
    "BMI <= 25": lambda frame: frame["BMI"] <= 25,
}
pattern_trace = pd.DataFrame({
    "Train_Position": np.arange(len(X_train)),
    "Source_Row_Index": train_idx,
    "Actual": y_train.to_numpy(dtype=int),
})
match_columns = []
for pattern_number, (condition, rule) in enumerate(
    PATTERN_RULES.items(), start=1
):
    column = f"Match_FN_P{pattern_number}"
    pattern_trace[column] = rule(X_train).to_numpy(dtype=int)
    match_columns.append(column)
pattern_trace["Pattern_Match_Count"] = (
    pattern_trace[match_columns].sum(axis=1).astype(int)
)

positive = pattern_trace.Actual.eq(1)
exactly_two = positive & pattern_trace.Pattern_Match_Count.eq(2)
at_least_three = positive & pattern_trace.Pattern_Match_Count.ge(3)
pattern_trace["Targeted_Positive"] = (
    positive & pattern_trace.Pattern_Match_Count.ge(2)
).astype(int)

weight_table = pattern_trace[
    ["Train_Position", "Source_Row_Index"]
].copy()
for scheme, two_weight, three_weight in [
    ("Mild", 1.10, 1.15),
    ("Balanced", 1.15, 1.25),
    ("High", 1.20, 1.35),
]:
    values = np.ones(len(weight_table), dtype=float)
    values[exactly_two] = two_weight
    values[at_least_three] = three_weight
    weight_table[scheme] = values

regenerated_weights_path = (
    OOF_PREDICTIONS_DIR / "lice_sample_weights_regenerated.csv.gz"
)
write_deterministic_gzip_csv(
    weight_table,
    regenerated_weights_path,
)
write_deterministic_gzip_csv(
    pattern_trace,
    OOF_PREDICTIONS_DIR / "lice_pattern_match_trace.csv.gz",
)

weighted_rows = int((weight_table.Balanced > 1.0).sum())
assert weighted_rows == 7174
EXPECTED_WEIGHTS_SHA256 = (
    "812aa92982d1c1264efd82b821eb9e7b"
    "af9d35e900517cc674ad83e504d5d607"
)
released_weights_path = INPUT_DIR / "lice_sample_weights.csv.gz"
observed_hash = file_sha256(released_weights_path)
assert observed_hash == EXPECTED_WEIGHTS_SHA256
released_weights = pd.read_csv(released_weights_path)
regenerated_weights = pd.read_csv(regenerated_weights_path)
pd.testing.assert_frame_equal(
    regenerated_weights,
    released_weights,
    check_dtype=True,
    check_exact=True,
)

weight_summary = pd.DataFrame([
    {
        "Scheme": scheme,
        "Rows": len(weight_table),
        "Weighted_Rows": int((weight_table[scheme] > 1.0).sum()),
        "Minimum": weight_table[scheme].min(),
        "Maximum": weight_table[scheme].max(),
        "Mean": weight_table[scheme].mean(),
    }
    for scheme in ["Mild", "Balanced", "High"]
])
weight_summary.to_csv(
    ABLATION_RESULTS_DIR / "sample_weight_summary_full_precision.csv",
    index=False,
    float_format="%.17g",
)
display(weight_summary)
print("All regenerated weight rows and values match the release artifact.")
print("Released weight artifact SHA-256:", observed_hash)


## 9. Interaction features and ablation inputs


In [ ]:
INTERACTION_DEFINITIONS = {
    "HighBP_HighChol": ("HighBP", "HighChol"),
    "HighBP_GenHlth": ("HighBP", "GenHlth"),
    "HighChol_GenHlth": ("HighChol", "GenHlth"),
    "BMI_GenHlth": ("BMI", "GenHlth"),
}
ABLATION_CONFIGURATIONS = {
    "Baseline": {"interactions": False, "weights": None},
    "M1-Interactions": {"interactions": True, "weights": None},
    "M2-Mild": {"interactions": False, "weights": "Mild"},
    "M2-Balanced": {"interactions": False, "weights": "Balanced"},
    "M2-High": {"interactions": False, "weights": "High"},
    "M3-Mild": {"interactions": True, "weights": "Mild"},
    "M3-Balanced": {"interactions": True, "weights": "Balanced"},
    "M3-High": {"interactions": True, "weights": "High"},
}

weight_record = {
    "random_seed": RANDOM_SEED,
    "oof_folds": OOF_FOLDS,
    "decision_threshold": DECISION_THRESHOLD,
    "model_parameters": MODEL_PARAMETERS,
    "lime_configuration": LIME_CONFIGURATION,
    "dice_configuration": DICE_CONFIGURATION,
    "feature_definitions": FEATURE_DEFINITIONS,
    "pattern_rules": list(PATTERN_RULES),
    "interaction_definitions": INTERACTION_DEFINITIONS,
    "ablation_configurations": ABLATION_CONFIGURATIONS,
    "released_weight_sha256": file_sha256(
        INPUT_DIR / "lice_sample_weights.csv.gz"
    ),
    "regenerated_weight_file": str(
        regenerated_weights_path.relative_to(PROJECT_ROOT)
    ),
}
with (ABLATION_RESULTS_DIR / "oof_explanation_and_weight_record.json").open("w") as stream:
    json.dump(weight_record, stream, indent=2)

print("OOF predictions and sample-weight identification completed.")
